In [ ]:
import pandas as pd
import sqlite3

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

paths = {
    "customers": PROJECT_ROOT / "data" / "olist_customers_dataset.xlsx",
    "geolocation": PROJECT_ROOT / "data" / "olist_geolocation_dataset.xlsx",
    "order_items": PROJECT_ROOT / "data" / "olist_order_items_dataset.xlsx",
    "order_payments": PROJECT_ROOT / "data" / "olist_order_payments_dataset.xlsx",
    "order_reviews": PROJECT_ROOT / "data" / "olist_order_reviews_dataset.xlsx",
    "orders": PROJECT_ROOT / "data" / "olist_orders_dataset.xlsx",
    "products": PROJECT_ROOT / "data" / "olist_products_dataset.xlsx",
    "sellers": PROJECT_ROOT / "data" / "olist_sellers_dataset.xlsx",
    "category_translation": PROJECT_ROOT / "data" / "product_category_name_translation.xlsx",
}

dataframes = {}


In [ ]:
conn = sqlite3.connect(":memory:")

In [ ]:
dataframes = {name: pd.read_excel(path) for name, path in paths.items()}

In [ ]:
from datetime import datetime

# Some Excel exports auto-format decimal monetary values as dates.
# Example: 26.08 can become 2026-08-26. Convert those cells back to numeric values.
def excel_date_number_to_float(value):
    if isinstance(value, datetime):
        if value.year == 2026:
            return value.day + value.month / 100
        return (value.year - 2000) + value.month / 100
    return value

numeric_columns = {
    "order_items": ["price", "freight_value"],
    "order_payments": ["payment_value"],
    "products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    ],
    "geolocation": ["geolocation_lat", "geolocation_lng"],
}

for table_name, columns in numeric_columns.items():
    for column in columns:
        if column in dataframes[table_name].columns:
            dataframes[table_name][column] = (
                dataframes[table_name][column]
                .map(excel_date_number_to_float)
                .pipe(pd.to_numeric, errors="coerce")
            )


In [ ]:
for name, path in paths.items():
    dataframes[name].to_sql(name, conn, index=False, if_exists="replace")


In [ ]:
print('Table names:')
for name, df in dataframes.items():
    print(name, df.columns)

In [ ]:
# Relationships between tables:

# orders.customer_id → customers.customer_id
# order_items.order_id → orders.order_id
# order_items.product_id → products.product_id
# order_items.seller_id → sellers.seller_id
# order_payments.order_id → orders.order_id
# order_reviews.order_id → orders.order_id
# products.product_category_name → category_translation.product_category_name

## Exploratory Data Analysis

### Block 1 - General Business Overview
1. How many total orders are in the dataset?
2. How many unique customers are there?
3. How many sellers operate on the platform?
4. How many unique products are sold?
5. What is the average order value?

### Block 2 - Sales Dynamics
6. How did order volume change by month?
7. Is there seasonality in sales?
8. Which month has the highest order volume?
9. How does GMV (Gross Merchandise Value) change over time?

### Block 3 - Customer Behavior
10. How many orders does the average customer place?
11. What share of customers make repeat purchases?
12. What is the average time between repeat orders?
13. What is Customer Lifetime Value (CLV)?

### Block 4 - Payment Analysis
14. Which payment methods are most popular?
15. What is the average order value by payment method?
16. How much revenue does each payment method generate?

### Block 5 - Customer Reviews
17. What is the average order rating?
18. What share of reviews are negative (scores 1-2)?
19. Does delivery time affect review score?
20. Which product categories receive the worst reviews?

### Block 6 - Logistics
21. How long does delivery take on average?
22. How often are deliveries late?
23. Which states have the highest late-delivery risk?
24. Do delivery delays affect ratings?

### Block 7 - Seller Analysis
25. How many orders does the average seller receive?
26. Who are the top 10 sellers by order volume?
27. Is seller quality reflected in review scores?
28. Which sellers have the highest negative-review risk?

### Block 8 - Product Category Analysis
29. Which product categories sell best?
30. Which categories generate the most revenue?
31. Which categories have the weakest reviews?
32. Which categories have the longest delivery times?

### Block 9 - Geographic Analysis
33. Which states have the most customers?
34. Which states have the most sellers?
35. How does average order value vary by region?
36. Is distance related to delivery time?

### Block 10 - Business Risks
37. What share of orders are canceled?
38. Are cancellations concentrated among specific sellers?
39. Does a long delivery window affect cancellations?
40. Which categories have the highest composite business risk?

## Block 1 - General Business Overview
1. How many total orders are in the dataset?
2. How many unique customers are there?
3. How many sellers operate on the platform?
4. How many unique products are sold?
5. What is the average order value?

In [ ]:
# 1. How many total orders are in the dataset?
query = """
SELECT
    COUNT(DISTINCT(order_id))  AS cnt_orders
FROM
    orders
WHERE order_id IS NOT NULL
LIMIT 10;
"""
pd.read_sql_query(query, conn)

In [ ]:
# 2. How many unique customers are there?
query = """
SELECT
    COUNT(DISTINCT(customer_id)) AS uniq_customer
FROM
    orders
WHERE customer_id IS NOT NULL
LIMIT 10;
"""
pd.read_sql_query(query, conn)

In [ ]:
# 3. How many sellers operate on the platform?
query = """
SELECT
    COUNT(DISTINCT(seller_id)) AS cnt_sellers
FROM
    sellers
WHERE seller_id IS NOT NULL
LIMIT 10;
"""
pd.read_sql_query(query, conn)

In [ ]:
# 4. How many unique products are sold?
query = """
SELECT
    COUNT(DISTINCT product_id) AS unique_products_sold
FROM order_items
WHERE product_id IS NOT NULL;
"""
pd.read_sql_query(query, conn)


In [ ]:
# 5. What is the average order value?
query = """
SELECT
    ROUND(AVG(order_total), 2) AS avg_order_value
FROM (
    SELECT
        order_id,
        SUM(payment_value) AS order_total
    FROM order_payments
    WHERE payment_value IS NOT NULL
    GROUP BY order_id
);
"""
pd.read_sql_query(query, conn)


## Block 2 - Sales Dynamics
6. How did order volume change by month?
7. Is there seasonality in sales?
8. Which month has the highest order volume?
9. How does GMV (Gross Merchandise Value) change over time?

In [ ]:
# 6. How did order volume change by month?
query = """
SELECT
    strftime('%Y-%m', order_purchase_timestamp) AS order_month,
    COUNT(*) AS orders_count
FROM
    orders
GROUP BY order_month 
ORDER BY order_month;
"""
pd.read_sql_query(query, conn)

### Key Takeaway
Order volume grew quickly throughout 2017, rising almost tenfold from January to November. The clear peak appears in November 2017, likely driven by Black Friday promotions. In 2018, demand stabilized around roughly 6.5k-7.2k monthly orders. September and October 2018 appear incomplete and should be excluded from trend interpretation.

In [ ]:
# 7. Is there seasonality in sales?
query = """
SELECT
    substr(year_month, 6, 2) AS month,
    ROUND(AVG(month_orders),2) AS avg_orders
FROM
(
    SELECT
        strftime('%Y-%m', order_purchase_timestamp) AS year_month,
        COUNT(*) AS month_orders
    FROM orders
    GROUP BY year_month
    HAVING COUNT(*) > 1000
)
GROUP BY month
ORDER BY month;
"""
pd.read_sql_query(query, conn)

### Key Takeaway
The data shows a visible seasonal pattern. Orders peak in November, likely because of Black Friday, and remain relatively strong in December because of holiday shopping. February and September are weaker months, while mid-year demand is more stable.

In [ ]:
# 8. Which month has the highest order volume?
query = """
SELECT
    substr(year_month, 6, 2) AS month,
    ROUND(AVG(month_orders),2) AS avg_orders
FROM
(
    SELECT
        strftime('%Y-%m', order_purchase_timestamp) AS year_month,
        COUNT(*) AS month_orders
    FROM orders
    GROUP BY year_month
    HAVING COUNT(*) > 1000
)
GROUP BY month
ORDER BY avg_orders DESC
LIMIT 1;
"""
pd.read_sql_query(query, conn)

### Key Takeaway
November has the highest average order volume, which is consistent with the Black Friday sales cycle.

In [ ]:
# 9. How does GMV (Gross Merchandise Value) change over time?
query = """
SELECT 
    strftime('%Y-%m', o.order_purchase_timestamp) AS year_month,
    ROUND(SUM(oi.price + oi.freight_value),2) AS gmv
    /* GMV total */
FROM
    orders o
JOIN order_items oi
ON o.order_id = oi.order_id
GROUP BY year_month
ORDER BY year_month; 
"""
pd.read_sql_query(query, conn)

### Key Takeaway
GMV started from a small base in October 2016 and accelerated sharply during 2017, growing almost every month. There is a short pre-season dip around September 2017, followed by a November 2017 peak that is likely connected to Black Friday demand.

## Block 3 - Customer Behavior
10. How many orders does the average customer place?
11. What share of customers make repeat purchases?
12. What is the average time between repeat orders?
13. What is Customer Lifetime Value (CLV)?

In [ ]:
# 10. How many orders does the average customer place?
query = """
SELECT
    ROUND(AVG(cnt_orders), 2) AS avg_ord_per_customer
FROM
(
SELECT
    customer_unique_id,
    COUNT(DISTINCT order_id) AS cnt_orders -- Number of orders per customer
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY customer_unique_id
)
"""
pd.read_sql_query(query, conn)


### Key Takeaway
The average customer places about 1.03 orders, which means the marketplace is dominated by one-time buyers and has a low retention baseline.

In [ ]:
# 11. What share of customers make repeat purchases?
query = """
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN cnt_orders > 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS repeat_purchase_rate
FROM
(
SELECT
    customer_unique_id,
    COUNT(DISTINCT order_id) AS cnt_orders
FROM customers c
JOIN orders o 
ON c.customer_id = o.customer_id
GROUP BY customer_unique_id
)
"""
pd.read_sql_query(query, conn)



### Key Takeaway
Repeat purchase rate is very low. Most customers buy once and do not return, making retention one of the largest business opportunities in the dataset.

In [ ]:
# 12. What is the average time between repeat orders?
query = """
SELECT
    ROUND(AVG(days_between_orders),2) AS avg_days_between_orders
FROM
(
SELECT
    customer_unique_id,
    julianday(order_purchase_timestamp) -
    julianday(LAG(order_purchase_timestamp)
        OVER (
            PARTITION BY customer_unique_id
            ORDER BY order_purchase_timestamp
        )
    ) AS days_between_orders
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
)
WHERE days_between_orders IS NOT NULL;
""" 
pd.read_sql_query(query, conn)


### Key Takeaway
Repeat customers usually wait about 78 days before placing another order. That two-to-three-month cycle creates room for targeted reactivation campaigns, reminders, and post-purchase flows.

In [ ]:
# 13. What is Customer Lifetime Value (CLV)?
query = """
SELECT
    AVG(customer_revenue) AS avg_clv
FROM (
    SELECT
        c.customer_unique_id, 
        SUM(oi.price + oi.freight_value) AS customer_revenue
    FROM
        orders o    
    JOIN customers c 
        ON o.customer_id = c.customer_id
    JOIN order_items oi 
         ON oi.order_id = o.order_id
    GROUP BY c.customer_unique_id
);
"""
pd.read_sql_query(query, conn)

### Key Takeaway
Average CLV is close to the value of one order, so growth is driven more by acquisition than by retention. Marketing spend should stay below the expected order-level contribution margin, and the biggest upside is increasing repeat purchases.

### Recommended Actions
- Build retention flows with email marketing, personalized recommendations, vouchers, and post-purchase offers.
- Track CLV by acquisition channel before scaling paid campaigns.
- Use repeat-purchase incentives for customers who approach the 60-90 day window after their first order.

## Block 4 - Payment Analysis
14. Which payment methods are most popular?
15. What is the average order value by payment method?
16. How much revenue does each payment method generate?

In [ ]:
# 14. Which payment methods are most popular?
query = """
SELECT
    payment_type,
    COUNT(payment_type) AS cnt_type,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS percent 
FROM
    order_payments
WHERE payment_type IS NOT NULL
GROUP BY payment_type
ORDER BY cnt_type DESC;
"""
pd.read_sql_query(query, conn)

### Key Takeaway
Credit card is the primary payment method and the core revenue engine, so the business depends heavily on card payment reliability, gateway performance, and banking fees. Boleto remains important for Brazilian customers who do not rely on cards, while vouchers show that discount-driven behavior already exists. Debit card usage is limited.

### Recommended Actions
- Optimize the credit-card checkout flow for speed, reliability, and low failure rates.
- Keep local payment alternatives such as boleto available.
- Use vouchers selectively for retention and repeat-purchase campaigns.

In [ ]:
# 15. What is the average order value by payment method?
query = """
SELECT
    payment_type,
    ROUND(AVG(order_total), 2) AS avg_order_value
FROM (
    SELECT
        order_id,
        payment_type,
        SUM(payment_value) AS order_total
    FROM
        order_payments
    GROUP BY order_id, payment_type
)
GROUP BY payment_type
ORDER BY avg_order_value DESC;
 """
pd.read_sql_query(query, conn)

### Key Takeaway
Vouchers have the highest average order value but not the highest total revenue. They appear to encourage customers to add more to the basket, while credit cards remain the stable high-volume segment. Boleto and debit card purchases have lower AOV and may represent more price-sensitive customers.

### Recommended Actions
- Scale voucher campaigns carefully and monitor margin impact.
- Keep credit card performance as a priority because it is the revenue base.
- Test personalized discounts and threshold-based offers to increase basket size without eroding profit.

In [ ]:
# 16. How much revenue does each payment method generate?
query = """
SELECT
    payment_type,
    ROUND(SUM(order_total), 2) AS revenue_pay_type
FROM (
    SELECT
        order_id,
        payment_type,
        SUM(payment_value) AS order_total
    FROM
        order_payments
    GROUP BY order_id, payment_type
)
GROUP BY payment_type
ORDER BY revenue_pay_type DESC;
 """
pd.read_sql_query(query, conn)

### Key Takeaway
Credit cards generate most of the total revenue. Voucher orders have high AOV but low usage frequency, creating a gap between transaction size and total revenue impact. Boleto is still an important secondary method, while debit card usage is minimal.

## Block 5 - Customer Reviews
17. What is the average order rating?
18. What share of reviews are negative (scores 1-2)?
19. Does delivery time affect review score?
20. Which product categories receive the worst reviews?

In [ ]:
# 17. What is the average order rating?
query = """
SELECT
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM 
    order_reviews
WHERE review_score IS NOT NULL;
 """
pd.read_sql_query(query, conn)

In [ ]:
# 18. What share of reviews are negative (scores 1-2)?
query = """
SELECT
    SUM(percent) AS part_reviews_score_1_and_2
FROM
    (
    SELECT
        review_score,
        COUNT(review_score) AS cnt_review_score,
        ROUND(100.0 * COUNT(*)/ SUM(COUNT(*))  OVER(), 2) AS percent
    FROM 
        order_reviews
    WHERE
       review_score IS NOT NULL
    GROUP BY review_score
    ORDER BY review_score
    )
WHERE review_score <= 2 
 """
pd.read_sql_query(query, conn)

### Key Takeaway
A healthy average rating can hide a meaningful negative-review segment. The 1-2 score group is the most important signal for operational improvement.

### Recommended Actions
- Prioritize root-cause analysis for negative reviews by category, seller, delivery time, and payment flow.
- Reduce the negative-review share through delivery improvements and clearer customer expectations.
- Treat 1-2 score reviews as the primary source for quality-control actions.

In [ ]:
# 19. Does delivery time affect review score?
query = """
SELECT
    review_score,
    AVG(delivery_time) AS avg_delivery_time
FROM
    (
    SELECT
        julianday(order_delivered_customer_date) - julianday(order_approved_at) AS delivery_time,
        review_score  
    FROM 
        orders o
    JOIN order_reviews orr ON o.order_id = orr.order_id
    WHERE
        order_delivered_customer_date IS NOT NULL
        AND order_approved_at IS NOT NULL
    )
GROUP BY review_score
"""
pd.read_sql_query(query, conn)

### Key Takeaway
Longer delivery time is associated with lower review scores, making logistics one of the strongest drivers of customer satisfaction.

### Recommended Actions
- Improve delivery speed through better fulfillment, warehouse coverage, and carrier performance.
- Manage SLA targets closely and flag orders likely to exceed the delivery window.
- Show realistic ETA information and proactively warn customers when delays are expected.

In [ ]:
# 20. Which product categories receive the worst reviews?
query = """
SELECT
    ct.product_category_name_english,
    COUNT(*) AS review_count,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM
    products p
JOIN order_items oi
    ON p.product_id = oi.product_id
JOIN order_reviews orr
    ON oi.order_id = orr.order_id 
JOIN category_translation ct 
    ON p.product_category_name = ct.product_category_name
GROUP BY ct.product_category_name_english
HAVING COUNT(*) >= 30
ORDER BY avg_review_score ASC, review_count DESC
LIMIT 20;
"""
pd.read_sql_query(query, conn)


### Key Takeaway
The largest review risks are concentrated in categories such as office furniture, home comfort, and audio. Low average ratings combined with meaningful order volume create a large impact on marketplace trust. Furniture and complex/bulky products likely suffer from long delivery times, damage risk, and mismatches between expectation and reality.

### Recommended Actions
- Focus first on office furniture as the largest review pain point.
- Investigate delivery time, returns, and review text for bulky categories.
- Improve product content with more photos, dimensions, and usage instructions.
- Segment category issues by seller, delivery performance, and review score.

## Block 6 - Logistics
21. How long does delivery take on average?
22. How often are deliveries late?
23. Which states have the highest late-delivery risk?
24. Do delivery delays affect ratings?

In [ ]:
# 21. How long does delivery take on average?
query = """
SELECT
    AVG(julianday(order_delivered_customer_date) - julianday(order_approved_at)) AS delivery_time
FROM
    orders
"""
pd.read_sql_query(query, conn)



In [ ]:
# 22. How often are deliveries late?
query = """
SELECT
    ROUND(
        100.0 * SUM(
            CASE 
                WHEN julianday(order_delivered_customer_date)
                    > julianday(order_estimated_delivery_date)
                THEN 1 ELSE 0
            END
        ) / COUNT(*), 2
    ) AS delayed_percent
FROM
    orders
WHERE 
    order_delivered_customer_date IS NOT NULL
    AND order_estimated_delivery_date IS NOT NULL
"""
pd.read_sql_query(query, conn)


### Key Takeaway
About 8.1% of delivered orders arrive later than the estimated delivery date.

In [ ]:
# 23. Which states have the highest late-delivery risk?
query = """
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    SUM(CASE WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1 ELSE 0 END) AS delayed_orders,
    ROUND(100.0 * SUM(CASE WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1 ELSE 0 END) / COUNT(DISTINCT o.order_id), 2) AS delayed_rate_pct,
    ROUND(AVG(CASE WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date)
        THEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) END), 2) AS avg_delay_days_late_only
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY c.customer_state
HAVING delivered_orders >= 100
ORDER BY delayed_rate_pct DESC, avg_delay_days_late_only DESC;
"""
pd.read_sql_query(query, conn)


### Key Takeaway
Late-delivery risk is most important where high order volume overlaps with high delay rates. Some remote states show large delays but low volume, while key states also show meaningful delay exposure. This suggests both geographic complexity and operational execution issues.

### Recommended Actions
- Prioritize delivery improvement in high-volume states such as SP, RJ, MG, RS, and PR.
- Test better carrier routing, warehouse placement, and region-specific fulfillment rules.
- Use region-specific SLA and ETA communication instead of a single marketplace-wide promise.
- Audit carrier performance in states with repeated delay issues.

In [ ]:
# 24. Which states and categories have the highest late-delivery exposure?
query = """
SELECT
    c.customer_state,
    (julianday(o.order_estimated_delivery_date)- julianday(o.order_delivered_customer_date)) AS avg_delay_days
FROM 
    orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE
    o.order_delivered_customer_date IS NOT NULL
    AND o.order_estimated_delivery_date IS NOT NULL
    AND c.customer_state = 'SP'
GROUP BY c.customer_state
ORDER BY avg_delay_days DESC;
"""
pd.read_sql_query(query, conn)


## Block 7 - Seller Analysis
25. How many orders does the average seller receive?
26. Who are the top 10 sellers by order volume?
27. Is seller quality reflected in review scores?
28. Which sellers have the highest negative-review risk?

In [ ]:
# 25. How many orders does the average seller receive?
query = """
SELECT
    ROUND(AVG(orders_count), 2) AS avg_orders_per_seller,
    ROUND(AVG(items_count), 2) AS avg_items_per_seller
FROM (
    SELECT
        seller_id,
        COUNT(DISTINCT order_id) AS orders_count,
        COUNT(*) AS items_count
    FROM order_items
    GROUP BY seller_id
);
"""
pd.read_sql_query(query, conn)


In [ ]:
# 26. Who are the top 10 sellers by order volume?
query = """
SELECT
    s.seller_id,
    COUNT(*) AS count_orders
FROM
    order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
GROUP BY s.seller_id
ORDER BY count_orders DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


In [ ]:
# 27. Is seller quality reflected in review scores?
query = """
SELECT
    MIN(avg_reviews_score) AS min_avg,
    MAX(avg_reviews_score) AS max_avg,
    MAX(avg_reviews_score) - MIN(avg_reviews_score) AS difference_avg
FROM
(
SELECT
    s.seller_id,
    COUNT(DISTINCT oi.order_id) AS count_orders,
    ROUND(AVG(orr.review_score), 3) AS avg_reviews_score,
    COUNT(orr.review_score) AS count_reviews
FROM
    order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
JOIN order_reviews orr ON orr.order_id = oi.order_id
GROUP BY s.seller_id
HAVING COUNT(DISTINCT oi.order_id) > 50
ORDER BY avg_reviews_score ASC
)
"""
pd.read_sql_query(query, conn)



### Key Takeaway
Review scores vary meaningfully by seller, which indicates that seller quality is part of the customer-experience problem.

### Recommended Actions
- Introduce seller scorecards and SLA monitoring.
- Limit visibility for consistently weak sellers.
- Highlight top sellers in search and merchandising surfaces.

In [ ]:
# 28. Which sellers have the most negative reviews?
query = """

SELECT
    s.seller_id,
    COUNT(DISTINCT CASE WHEN orr.review_score <= 2  THEN orr.review_id END) AS count_bad_reviews,
    COUNT(DISTINCT orr.review_id) AS total_reviews,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN orr.review_score <= 2  THEN orr.review_id END)/COUNT(DISTINCT orr.review_id),2) AS bad_review_rate
FROM
    order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
JOIN order_reviews orr ON orr.order_id = oi.order_id
JOIN orders o ON oi.order_id = o.order_id
GROUP BY s.seller_id
HAVING COUNT(DISTINCT oi.order_id) > 50
ORDER BY bad_review_rate DESC
LIMIT 20;
"""
pd.read_sql_query(query, conn)


### Key Takeaway
Some sellers have extremely high negative-review rates, and a larger group sits in the 30-40% risk range. The issue is not simply seller size; it is seller quality and operational control.

### Recommended Actions
- Segment sellers by negative-review rate, delay rate, and cancellation rate.
- Pause or review sellers with the highest bad-review rates.
- Diagnose whether negative reviews come from logistics, product quality, or inaccurate descriptions.
- Reduce weak-seller visibility and add monitoring for improving or worsening trends.

## Block 8 - Product Category Analysis
29. Which product categories sell best?
30. Which categories generate the most revenue?
31. Which categories have the weakest reviews?
32. Which categories have the longest delivery times?

In [ ]:
# 29. Which product categories sell best?
query = """
SELECT
    ct.product_category_name_english,
    COUNT(DISTINCT oi.order_id) AS count_order
FROM
    products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
WHERE ct.product_category_name_english IS NOT NULL
GROUP BY ct.product_category_name_english
ORDER BY count_order DESC
LIMIT 20
"""

pd.read_sql_query(query, conn)


### Key Takeaway
Top categories such as bed_bath_table, health_beauty, and sports_leisure reflect broad everyday demand. Computers/accessories sell more frequently than electronics, which fits a pattern where customers buy lower-priced add-ons more often than expensive devices. Niche categories such as consoles_games and office_furniture have lower frequency and may involve narrower demand or higher-ticket purchases.

### Recommended Actions
- Compare volume with margin before deciding where to invest.
- Protect inventory availability in high-demand categories.
- Use cross-sell tactics, such as decor or furniture recommendations after bed and bath purchases.
- Treat electronics as a separate strategy with a longer decision cycle and more trust-building content.

In [ ]:
# 30. Which categories generate the most revenue?
query = """
SELECT
    ct.product_category_name_english,
    COUNT(DISTINCT oi.order_id) AS count_orders,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue,
    AVG(oi.price) AS avg_price
FROM
    products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
WHERE ct.product_category_name_english IS NOT NULL
GROUP BY ct.product_category_name_english
ORDER BY revenue DESC
LIMIT 20
"""
pd.read_sql_query(query, conn)


### Key Takeaway
Revenue is concentrated in practical, everyday categories. Lower electronics revenue may reflect lower demand, stronger competition, or a different buying cycle. Accessories selling more often than electronics is a classic e-commerce pattern.

### Recommended Actions
- Focus growth on high-volume categories while checking margin quality.
- Build cross-sell paths from bed and bath to decor or housewares, and from computers to accessories.
- Avoid scaling electronics blindly; use brand trust, warranty, and product-content improvements.
- Keep stock availability high in top-demand categories.

In [ ]:
# 31. Which categories have the weakest reviews?
query = """
SELECT
    ct.product_category_name_english,
    ROUND(AVG(orr.review_score), 2) AS avg_review,
    COUNT(DISTINCT orr.review_id) AS count_score,
    COUNT(DISTINCT CASE WHEN orr.review_score <= 2 THEN orr.review_id END) AS count_bad_review,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN orr.review_score <= 2 THEN orr.review_id END)/ COUNT(DISTINCT orr.review_id), 2) AS percent_negative_review
FROM
    products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
JOIN order_reviews orr ON orr.order_id = oi.order_id
GROUP BY ct.product_category_name_english 
HAVING COUNT(orr.review_score) > 50
ORDER BY percent_negative_review DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)


### Key Takeaway
Several categories show structurally high negative-review rates. Office furniture is especially risky because it combines high review volume with a high negative-review share. Large core categories such as bed_bath_table, furniture_decor, and computers_accessories also show elevated negative-review exposure, so the issue is not limited to small niches.

### Recommended Actions
- Prioritize categories with both high volume and high negative-review share.
- Run root-cause analysis across delivery, product quality, expectations, and seller behavior.
- Add product UX signals such as review quality, negative-review share, and verified customer feedback.
- Treat category quality as a retention and trust risk, not only an operations issue.

In [ ]:
# 32. Which categories have the longest delivery times?
query = """
SELECT
    ct.product_category_name_english,
    ROUND(AVG(julianday(order_delivered_customer_date) - julianday(order_approved_at)), 2) AS avg_delivery_time,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM
    products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
JOIN orders o ON o.order_id = oi.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_approved_at IS NOT NULL
GROUP BY ct.product_category_name_english 
HAVING COUNT(DISTINCT o.order_id) > 50
ORDER BY avg_delivery_time DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)


### Key Takeaway
Long delivery times are concentrated in furniture, bulky products, and categories that are harder to move through the logistics network.

### Recommended Actions
- Make office furniture the first logistics-improvement priority.
- Review seller performance, warehouse coverage, and carrier behavior for slow categories.
- Separate category-level delivery promises for fast accessories versus slow bulky goods.
- Show realistic ETA and tracking information when delivery is expected to be slow.

## Block 9 - Geographic Analysis
33. Which states have the most customers?
34. Which states have the most sellers?
35. How does average order value vary by region?
36. Is distance related to delivery time?

In [ ]:
# 33. Which states have the most customers?
query = """
SELECT
    customer_state,
    COUNT(DISTINCT customer_unique_id) AS count_customers
FROM
    customers
GROUP BY customer_state
ORDER BY count_customers DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)


### Key Takeaway
SP is the dominant customer market and the core demand center. RJ and MG form the next major demand tier, while RS and PR are smaller but still meaningful. Smaller states create a long tail that may offer growth potential but should be tested carefully.

### Recommended Actions
- Keep SP as the main operational and marketing focus.
- Scale RJ and MG with targeted acquisition and logistics improvements.
- Test campaigns in mid-sized states such as RS and PR.
- Investigate whether smaller markets are limited by delivery, awareness, assortment, or local demand.

In [ ]:
# 34. Which states have the most sellers?
query = """
SELECT
    seller_state,
    COUNT(DISTINCT seller_id) AS count_sellers
FROM
    sellers
GROUP BY seller_state
ORDER BY count_sellers DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)


### Key Takeaway
SP also dominates seller supply, but several high-demand states have fewer sellers relative to customer volume. RJ, MG, and BA show potential seller-supply gaps.

### Recommended Actions
- Recruit sellers in deficit regions, especially RJ, MG, and BA.
- Lower onboarding barriers and test seller incentives in underserved states.
- Use seller quality controls in saturated markets to avoid weak competition.
- Improve regional logistics with local warehouses or fulfillment centers where demand supports it.

In [ ]:
# 35. How does average order value vary by region?
query = """
SELECT
    c.customer_state,
    ROUND(AVG(order_total), 2) AS avg_order_value,
    COUNT(*) AS orders_count,
    COUNT(DISTINCT customer_unique_id) AS count_customers
FROM
(
    SELECT
        o.order_id,
        o.customer_id,
        SUM(oi.price + oi.freight_value) AS order_total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id, o.customer_id
) t
JOIN customers c ON t.customer_id = c.customer_id
GROUP BY c.customer_state
HAVING COUNT(*) > 100
ORDER BY avg_order_value DESC;
"""

pd.read_sql_query(query, conn)


### Key Takeaway
Average order value differs meaningfully across states. SP behaves like a high-volume, lower-AOV market, while some smaller states show premium-order behavior. RJ and MG look like balanced regions with both demand and monetization potential.

### Recommended Actions
- Raise SP AOV with bundles, upsell, and cross-sell tactics.
- Scale RJ and MG with broader assortment and targeted marketing.
- Build separate premium positioning for high-AOV states such as MS and DF.
- Diagnose weak regions through logistics, income, assortment, and delivery availability.

In [ ]:
# 36. Is distance related to delivery time?
query = """
WITH geo AS (
    SELECT
        geolocation_zip_code_prefix,
        AVG(geolocation_lat) AS lat,
        AVG(geolocation_lng) AS lng
    FROM geolocation
    GROUP BY geolocation_zip_code_prefix
), order_distance AS (
    SELECT
        o.order_id,
        ABS(gc.lat - gs.lat) + ABS(gc.lng - gs.lng) AS distance_proxy,
        julianday(o.order_delivered_customer_date) - julianday(o.order_approved_at) AS delivery_days
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN sellers s ON oi.seller_id = s.seller_id
    JOIN geo gc ON c.customer_zip_code_prefix = gc.geolocation_zip_code_prefix
    JOIN geo gs ON s.seller_zip_code_prefix = gs.geolocation_zip_code_prefix
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_approved_at IS NOT NULL
)
SELECT
    ROUND(AVG(distance_proxy), 4) AS avg_distance_proxy,
    ROUND(AVG(delivery_days), 2) AS avg_delivery_days
FROM order_distance;
"""
pd.read_sql_query(query, conn)


### Recommended Actions
- Make logistics optimization the top priority when average delivery time is high.
- Check processing time, dispatch time, warehouse delays, and carrier behavior separately.
- Reduce last-mile distance with local sellers, regional warehouses, or fulfillment centers.
- Segment delivery promises by distance and region to improve the customer experience.

## Block 10 - Business Risks
37. What share of orders are canceled?
38. Are cancellations concentrated among specific sellers?
39. Does a long delivery window affect cancellations?
40. Which categories have the highest composite business risk?

In [ ]:
# 37. What share of orders are canceled?
query = """
SELECT
    COUNT(*) AS count_all,
    SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS count_canceled,
    SUM(100.0 * CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) / COUNT(*) AS percent_canceled
FROM
    orders
"""

pd.read_sql_query(query, conn)


In [ ]:
# 38. Are cancellations concentrated among specific sellers?
query = """
SELECT
    oi.seller_id,
    COUNT(DISTINCT CASE WHEN o.order_status = 'canceled' THEN o.order_id END) AS count_canceled
FROM
    order_items oi
JOIN orders o ON oi.order_id = o.order_id
GROUP BY oi.seller_id
HAVING count_canceled > 1
ORDER BY count_canceled DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)


In [ ]:
# 39. Does a long delivery window affect cancellations?
query = """
SELECT
    CASE WHEN (julianday(order_estimated_delivery_date) - julianday(order_purchase_timestamp)) > 7 THEN 'long_delivery' ELSE 'short_delivery' END AS delivery_group,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
    ROUND(100.0 * SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancel_rate
FROM orders
GROUP BY delivery_group
"""

pd.read_sql_query(query, conn)


In [ ]:
# 40. Which categories have the highest composite business risk?
query = """
SELECT
    pct.product_category_name_english AS category,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
    ROUND(100.0 * SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END)/ COUNT(*), 2) AS cancel_rate
FROM
    order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation pct ON p.product_category_name = pct.product_category_name
GROUP BY category
HAVING total_orders > 50
ORDER BY cancel_rate DESC
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    *
FROM orders o
JOIN order_items oi 
    ON o.order_id = oi.order_id
JOIN sellers s 
    ON oi.seller_id = s.seller_id
JOIN customers c 
    ON o.customer_id = c.customer_id
JOIN geolocation gc 
    ON c.customer_zip_code_prefix = gc.geolocation_zip_code_prefix
JOIN geolocation gs 
    ON s.seller_zip_code_prefix = gs.geolocation_zip_code_prefix
JOIN products p
    ON p.product_id = oi.product_id
JOIN order_payments op
    ON o.order_id = op.order_id
JOIN category_translation ct
    ON ct.product_category_name = p.product_category_name
LIMIT 5;
"""
pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    *
FROM products

"""
pd.read_sql_query(query, conn)
